# Week 1 — ECAPA-TDNN Baseline on VoxCeleb1 (Verification + Identification)

Loads a pretrained ECAPA-TDNN (SpeechBrain) and establishes **two** baselines before any fine-tuning:
1. **Speaker Verification (SV)** — EER / minDCF on the official `veri_test2.txt` trial pairs.
2. **Speaker Identification (SID)** — closed-set top-1 accuracy on the official `iden_split.txt` protocol (1,251 speakers), via nearest-centroid classification on embeddings.

Uses your Kaggle dataset layout:
- `sabahesaraki/voxceleb-1-dataset` → list/split files (`veri_test2.txt`, `iden_split.txt`, etc.)
- `kryakrya/voxceleb1train` and `kryakrya/voxceleb1test` → the actual audio (1,211 dev-speaker folders + 40 test-speaker folders)

Make sure GPU + Internet are ON in notebook settings (Internet is needed once, to pull the pretrained weights from HuggingFace).

In [ ]:
!pip install -q speechbrain


In [ ]:
import os, random, pickle
from pathlib import Path
import torch
import torchaudio
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:
# ---- Paths (from your Kaggle setup) ----
path_split = Path("/kaggle/input/datasets/sabahesaraki/voxceleb-1-dataset")

path_data = Path("/kaggle/input/datasets/kryakrya")
path_data_train = path_data.joinpath("voxceleb1train/wav")   # 1,211 dev-speaker folders
path_data_test  = path_data.joinpath("voxceleb1test/wav")    # 40 test-speaker folders

VERI_TEST_PATH = path_split / "veri_test2.txt"
IDEN_SPLIT_PATH = path_split / "iden_split.txt"

assert VERI_TEST_PATH.exists(), f"Missing {VERI_TEST_PATH}"
assert IDEN_SPLIT_PATH.exists(), f"Missing {IDEN_SPLIT_PATH}"
print("Found list files OK.")

def resolve_wav(rel_path: str) -> Path:
    """A relative path like 'id10001/1zcIwhmdeo4/00001.wav' can live in either
    the train (dev, 1211 spk) or test (40 spk) folder depending on the file's split.
    Check both."""
    p_test = path_data_test / rel_path
    if p_test.exists():
        return p_test
    p_train = path_data_train / rel_path
    if p_train.exists():
        return p_train
    raise FileNotFoundError(f"Could not find {rel_path} under train or test wav roots")

# Sanity check
print(len(list(path_data_train.iterdir())), "train speaker folders")
print(len(list(path_data_test.iterdir())), "test speaker folders")


In [ ]:
# ---- Load pretrained ECAPA-TDNN ----
from speechbrain.inference.speaker import EncoderClassifier

classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/kaggle/working/pretrained_ecapa",
    run_opts={"device": device}
)
print("Model loaded.")

embedding_cache = {}

@torch.no_grad()
def get_embedding(rel_path: str) -> torch.Tensor:
    if rel_path in embedding_cache:
        return embedding_cache[rel_path]
    full_path = resolve_wav(rel_path)
    signal, fs = torchaudio.load(str(full_path))
    if fs != 16000:
        signal = torchaudio.functional.resample(signal, fs, 16000)
    signal = signal.to(device)
    emb = classifier.encode_batch(signal).squeeze().cpu()
    embedding_cache[rel_path] = emb
    return emb

# Warm-up check
test_line = open(VERI_TEST_PATH).readline().split()
_ = get_embedding(test_line[1])
print("Embedding shape:", _.shape)


## Part 1 — Speaker Verification (EER / minDCF)

In [ ]:
trials = []
with open(VERI_TEST_PATH) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        label, p1, p2 = parts
        trials.append((int(label), p1, p2))
print("Total verification trial pairs:", len(trials))

N_TRIALS = 3000  # set to None for the full ~37,720 pairs (final reported number)
random.seed(42)
eval_trials = trials if N_TRIALS is None else random.sample(trials, N_TRIALS)
print("Evaluating on", len(eval_trials), "pairs")

unique_files = set()
for _, p1, p2 in eval_trials:
    unique_files.update([p1, p2])
print("Unique files to embed:", len(unique_files))
for f in tqdm(unique_files):
    get_embedding(f)


In [ ]:
scores, labels = [], []
for label, p1, p2 in tqdm(eval_trials):
    e1, e2 = get_embedding(p1), get_embedding(p2)
    sim = F.cosine_similarity(e1.unsqueeze(0), e2.unsqueeze(0)).item()
    scores.append(sim)
    labels.append(label)

scores = np.array(scores)
labels = np.array(labels)

from sklearn.metrics import roc_curve

def compute_eer(labels, scores):
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2, thresholds[idx], fpr, fnr

def compute_min_dcf(labels, scores, p_target=0.05, c_miss=1, c_fa=1):
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
    i = np.argmin(dcf)
    norm = min(c_miss * p_target, c_fa * (1 - p_target))
    return dcf[i] / norm, thresholds[i]

eer, eer_thresh, fpr, fnr = compute_eer(labels, scores)
min_dcf, dcf_thresh = compute_min_dcf(labels, scores)

print(f"Baseline SV  EER:    {eer*100:.2f}%  (threshold={eer_thresh:.4f})")
print(f"Baseline SV  minDCF: {min_dcf:.4f}  (p_target=0.05, threshold={dcf_thresh:.4f})")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.plot(fpr, fnr, label='ECAPA-TDNN (pretrained)')
plt.plot([0,1],[0,1],'--',color='gray')
i = np.nanargmin(np.abs(fpr-fnr))
plt.scatter([fpr[i]],[fnr[i]], color='red', zorder=5, label=f'EER={eer*100:.2f}%')
plt.xlabel('False Positive Rate'); plt.ylabel('False Negative Rate')
plt.title('Baseline Verification: FPR vs FNR'); plt.legend(); plt.grid(alpha=0.3)
plt.savefig('/kaggle/working/baseline_sv_det_curve.png', dpi=150, bbox_inches='tight')
plt.show()


## Part 2 — Speaker Identification (closed-set, 1,251 speakers)

Protocol: `iden_split.txt` has one line per utterance: `<split> <relative_path>`, split = 1 (train), 2 (val), 3 (test). Speaker label = the `idXXXXX` folder name in the path.

Method: build one **enrollment centroid** per speaker by averaging embeddings of their split-1 (train) utterances, then classify each split-3 (test) utterance by nearest centroid (cosine similarity). This mirrors exactly what your live app will do for enrolled users.

In [ ]:
iden_entries = []  # (split, rel_path, speaker_id)
with open(IDEN_SPLIT_PATH) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 2:
            continue
        split_label, rel_path = parts
        speaker_id = rel_path.split('/')[0]
        iden_entries.append((int(split_label), rel_path, speaker_id))

print("Total identification entries:", len(iden_entries))
train_entries = [e for e in iden_entries if e[0] == 1]
test_entries  = [e for e in iden_entries if e[0] == 3]
print("Train (split=1) utterances:", len(train_entries))
print("Test  (split=3) utterances:", len(test_entries))
print("Unique speakers in train:", len(set(e[2] for e in train_entries)))


In [ ]:
# Subsample for a manageable first run - raise these for your final reported number
N_ENROLL_PER_SPEAKER = 3   # utterances per speaker used to build the centroid
N_TEST_PER_SPEAKER   = 3   # test utterances per speaker to classify

random.seed(42)

by_speaker_train = defaultdict(list)
for split, rel_path, spk in train_entries:
    by_speaker_train[spk].append(rel_path)

by_speaker_test = defaultdict(list)
for split, rel_path, spk in test_entries:
    by_speaker_test[spk].append(rel_path)

enroll_set = {}
for spk, files in by_speaker_train.items():
    enroll_set[spk] = random.sample(files, min(N_ENROLL_PER_SPEAKER, len(files)))

eval_set = {}
for spk, files in by_speaker_test.items():
    if spk not in enroll_set:
        continue  # skip speakers with no enrollment data
    eval_set[spk] = random.sample(files, min(N_TEST_PER_SPEAKER, len(files)))

n_enroll_files = sum(len(v) for v in enroll_set.values())
n_eval_files = sum(len(v) for v in eval_set.values())
print(f"Enrolling {len(enroll_set)} speakers using {n_enroll_files} files")
print(f"Evaluating {len(eval_set)} speakers using {n_eval_files} files")


In [ ]:
# Build centroids
speaker_centroids = {}
for spk, files in tqdm(enroll_set.items(), desc="Building centroids"):
    embs = torch.stack([get_embedding(f) for f in files])
    centroid = F.normalize(embs.mean(dim=0), dim=0)
    speaker_centroids[spk] = centroid

centroid_ids = list(speaker_centroids.keys())
centroid_matrix = torch.stack([speaker_centroids[s] for s in centroid_ids])  # [num_speakers, dim]
print("Centroid matrix shape:", centroid_matrix.shape)


In [ ]:
# Classify each test utterance by nearest centroid (cosine similarity)
correct, total = 0, 0
top5_correct = 0

for spk, files in tqdm(eval_set.items(), desc="Classifying"):
    for f in files:
        emb = F.normalize(get_embedding(f), dim=0)
        sims = centroid_matrix @ emb  # cosine sim since both normalized
        pred_idx = torch.argmax(sims).item()
        pred_spk = centroid_ids[pred_idx]

        top5_idx = torch.topk(sims, k=min(5, len(centroid_ids))).indices.tolist()
        top5_spk = [centroid_ids[i] for i in top5_idx]

        total += 1
        if pred_spk == spk:
            correct += 1
        if spk in top5_spk:
            top5_correct += 1

top1_acc = correct / total
top5_acc = top5_correct / total
print(f"Baseline SID Top-1 accuracy: {top1_acc*100:.2f}%  ({correct}/{total})")
print(f"Baseline SID Top-5 accuracy: {top5_acc*100:.2f}%  ({top5_correct}/{total})")


In [ ]:
# ---- Save everything for the report and for later comparison against fine-tuned model ----
results = {
    'sv': {
        'n_trials': len(eval_trials),
        'eer': float(eer),
        'eer_threshold': float(eer_thresh),
        'min_dcf': float(min_dcf),
        'min_dcf_threshold': float(dcf_thresh),
        'scores': scores,
        'labels': labels,
    },
    'sid': {
        'n_speakers_enrolled': len(enroll_set),
        'n_test_utterances': total,
        'top1_acc': float(top1_acc),
        'top5_acc': float(top5_acc),
        'n_enroll_per_speaker': N_ENROLL_PER_SPEAKER,
        'n_test_per_speaker': N_TEST_PER_SPEAKER,
    }
}

with open('/kaggle/working/baseline_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("=== SUMMARY FOR REPORT ===")
print(f"Model: speechbrain/spkrec-ecapa-voxceleb (pretrained, no fine-tuning)")
print(f"SV  - EER: {eer*100:.2f}%  |  minDCF: {min_dcf:.4f}  (n={len(eval_trials)} trial pairs)")
print(f"SID - Top-1: {top1_acc*100:.2f}%  |  Top-5: {top5_acc*100:.2f}%  (n={total} test utterances, {len(enroll_set)} speakers)")


## Next steps (Week 2)

1. Re-run Part 1 with `N_TRIALS = None` (full trial set) and Part 2 with larger `N_ENROLL_PER_SPEAKER` / `N_TEST_PER_SPEAKER` — record these as your **official baseline numbers** in the report.
2. Prepare a train/val split from the 1,211-speaker dev set (`path_data_train`) for fine-tuning.
3. Fine-tune ECAPA-TDNN (SpeechBrain's `recipes/VoxCeleb/SpeakerRec/` recipe is a good starting point) — additive-margin softmax on speaker ID.
4. Re-run **both Part 1 and Part 2** against the fine-tuned checkpoint's embeddings and compare against this baseline. That before/after comparison (SV: EER/minDCF, SID: top-1/top-5) is your Requirement 1 headline result.
5. Note in the report: the SID numbers here use the *same* identities the pretrained model saw during its original training, so frame this as an embedding-quality/enrollment-pipeline demonstration rather than a generalization claim — the SV number (unseen test speakers) is the fairer generalization metric.
